# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIRˆ2 colorectal cancer dataset specified via a Croissant schema using the `mlcroissant` Python library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

We load the dataset metadata and records using `mlcroissant`. Here, the dataset metadata provides useful information on the dataset contents, fields, origin, and scope.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Metadata as JSON
metadata = dataset.metadata.to_json()

print("Dataset name:", metadata.get("name"))
print("Description:", metadata.get("description"))
print("Date Published:", metadata.get("datePublished"))
print("Identifier:", metadata.get("identifier"))
print("License:", metadata.get("license"))

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Record sets are the main containers for tabular data in Croissant datasets. Each record set contains references to fields with unique `@id` values.

In [ ]:
# Obtain the available record sets and their @id
record_sets = dataset.record_sets()

print(f"Number of record sets: {len(record_sets)}")
for rs in record_sets:
    print(f"Record set @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")
    print(f"  Description: {rs.get('description', 'N/A')}")
    print(f"  Number of fields: {len(rs.get('field', []))}")
    # Print field @ids
    for field in rs.get('field', []):
        print(f"    Field @id: {field['@id']} -- Label: {field.get('label', field.get('name', 'N/A'))} -- Type: {field.get('dataType', 'N/A')}")
    print()
# (Optional) Print an example record
if record_sets:
    example_record_set_id = record_sets[0]['@id']
    for i, record in enumerate(dataset.records(record_set=example_record_set_id)):
        print(f"Example record from {example_record_set_id}: {record}")
        if i >= 2:
            break

## 3. Data Extraction
Load the data from each record set into a DataFrame for further analysis.

All entity references use their unique `@id`.

In [ ]:
# List record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

# Load each record set into a DataFrame
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"DataFrame for {record_set_id} loaded: {df.shape[0]} rows, {df.shape[1]} columns")

# Show column names of first record set
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"Columns of {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Let's filter, normalize, and group records based on their numeric fields using field `@id`s. We demonstrate typical EDA steps: removing records below a threshold, normalizing a numeric column, and grouping data by categorical attributes.

In [ ]:
# Select main DataFrame
df = dataframes[main_record_set_id]

# Determine a numeric field and group field (using @id, could be e.g. age, interval_month, etc.)
numeric_field_id = None
group_field_id = None

# Try to choose numeric field from available ones
for col in df.columns:
    # Guess numeric by dtype or name
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
if not numeric_field_id:
    # Fall back to a name-based pattern if needed
    for col in df.columns:
        if "age" in col.lower() or "interval" in col.lower() or "months" in col.lower():
            numeric_field_id = col
            break

# Try to choose a group field
for col in df.columns:
    if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
        group_field_id = col
        break

if numeric_field_id:
    threshold = 10
    # Remove outliers/low values
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field found for analysis.")

## 5. Visualization
Visualize distributions and relationships in the dataset.

Example: Distribution of the primary numeric field, and optionally relationships by group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and df.shape[0] > 0:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id], kde=True, bins=15)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion

In this notebook, we loaded, explored, and visualized the FAIRˆ2 colorectal cancer dataset using the `mlcroissant` library. We examined metadata, reviewed available record sets and fields (referenced by their unique `@id`), filtered by numeric attributes, normalized values, grouped by key fields, and visualized main features. This preliminary EDA provides a starting point for deeper clinical or molecular analysis.

For further use, consult the Croissant schema and model documentation to guide downstream tasks such as prediction, stratification, or FAIR data integration.